<a href="https://colab.research.google.com/github/Naaao9999/shikoku-economic-analysis/blob/main/01_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01_data_preprocessing: 産業連関表の部門統合

## 1. 概要
本ノートブックでは、1985年から2005年までの計4時点（S60, H2, H7, H17）の地域間産業連関表を対象に、長期時系列分析を可能にするための部門統合を行います。

## 2. 主な処理ステップ
本工程では以下の4つのパイプラインを実装しています。

1. **部門統合 (Sector Integration)**:
   マッピング表に基づき、時点間で異なる産業分類を共通の部門体系へ統合します。これにより、20年間のシームレスな構造比較が可能になります。
2. **データの整合性検証**:
   列差分検出および統計的チェックを行い、次工程のAPL計算や階層ベイズモデルの精度を担保します。

## パス・ディレクトリ設定（GitHub構成用）

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

# ==========================================
# 1. パス・ディレクトリ設定
# ==========================================
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path(".")
RAW_DATA_DIR = BASE_DIR / "data" / "raw"
MAPPING_DIR = BASE_DIR / "data" / "mapping"
INTERIM_DIR = BASE_DIR / "data" / "interim"

# 保存先ディレクトリの自動作成
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# 分析対象の年次設定
YEARS_CONFIG = {
    "S60": {"raw": "S60.xlsx", "integrated": "S60_integrated.xlsx"},
    "H2":  {"raw": "H2.xlsx",  "integrated": "H2_integrated.xlsx"},
    "H7":  {"raw": "H7.xlsx",  "integrated": "H7_integrated.xlsx"},
    "H17": {"raw": "H17.xlsx", "integrated": "H17_integrated.xlsx"}
}

# 地域間部門統合

In [ ]:
def load_mapping_file(file_path): # マッピングファイルを読み込み
    try:
        df = pd.read_excel(file_path)
        return df
    except Exception as e:
        print(f"マッピングファイルの読み込みエラー: {e}")
        return None

def create_integration_operations(mapping_df, year_column):
    operations = []
    industries_to_remove = []  # 削除すべき産業リスト
    grouped_data = {}

    # 統合部門ごとに産業をグループ化
    for idx, row in mapping_df.iterrows():
        integrated_sector = row['統合部門']
        industry = row[year_column]

        # 統合部門が空で産業が存在する場合、削除リストに追加
        if pd.isna(integrated_sector) and not pd.isna(industry):
            if isinstance(industry, str):
                # 日本語のカンマで分割
                inds_to_remove = re.split('、|,', industry)
                for ind in inds_to_remove:
                    ind = ind.strip()
                    if ind and ind not in industries_to_remove:
                        industries_to_remove.append(ind)
            continue

        # 両方とも空の場合はスキップ
        if pd.isna(integrated_sector) or pd.isna(industry):
            continue

        if integrated_sector not in grouped_data:
            grouped_data[integrated_sector] = []

        # 日本語のカンマで分割
        if isinstance(industry, str):
            industries = re.split('、|,', industry)
            for ind in industries:
                ind = ind.strip()
                if ind and ind not in grouped_data[integrated_sector]:
                    grouped_data[integrated_sector].append(ind)

    # 全ての部門に対して操作を作成
    for integrated_sector, industries in grouped_data.items():
        operations.append({
            "industries_to_integrate": industries,
            "new_industry_name": integrated_sector
        })

    return operations, industries_to_remove

def remove_industries_row(df, region, industries_to_remove, region_col_idx=1, industry_col_idx=3, data_start_row=6):
    """行方向で不要な産業を削除"""
    # 結果用データフレーム
    result_df = df.copy()

    # 削除する行のインデックスを格納
    rows_to_drop = []

    # 削除対象の産業を検索
    for i in range(data_start_row, len(result_df)):
        if i >= len(result_df) or region_col_idx >= len(result_df.columns) or industry_col_idx >= len(result_df.columns):
            continue

        current_region = str(result_df.iloc[i, region_col_idx]) if pd.notna(result_df.iloc[i, region_col_idx]) else ""
        current_industry = str(result_df.iloc[i, industry_col_idx]) if pd.notna(result_df.iloc[i, industry_col_idx]) else ""

        if current_region == region and current_industry in industries_to_remove:
            rows_to_drop.append(i)

    # 対象行を削除してインデックスをリセット
    if rows_to_drop:
        result_df = result_df.drop(index=rows_to_drop).reset_index(drop=True)

    return result_df

def remove_industries_column(df, region, industries_to_remove, region_row_idx=3, industry_row_idx=5, data_start_col=4):
    """列方向で不要な産業を削除"""
    # 結果用データフレーム
    result_df = df.copy()

    # 削除する列のインデックスを格納
    cols_to_drop = []

    # 削除対象の産業を検索
    for i in range(data_start_col, len(result_df.columns)):
        if (i < len(result_df.columns) and
            region_row_idx < len(result_df) and
            industry_row_idx < len(result_df)):

            region_val = str(result_df.iloc[region_row_idx, i]) if pd.notna(result_df.iloc[region_row_idx, i]) else ""
            industry_val = str(result_df.iloc[industry_row_idx, i]) if pd.notna(result_df.iloc[industry_row_idx, i]) else ""

            if region_val == region and industry_val in industries_to_remove:
                cols_to_drop.append(i)

    # 対象列を削除
    if cols_to_drop:
        cols_to_keep = [i for i in range(len(result_df.columns)) if i not in cols_to_drop]
        result_df = result_df.iloc[:, cols_to_keep]

    return result_df

def integrate_industries_row(df, region, integration_operations, region_col_idx=1, industry_col_idx=3, data_start_row=6):
    """行方向で産業を統合（1対1の名称変更も含む）"""
    # 結果用データフレーム
    result_df = df.copy()

    # 各統合操作を処理
    for operation in integration_operations:
        industries_to_integrate = operation["industries_to_integrate"]
        new_industry_name = operation["new_industry_name"]

        # 地域と産業の条件に一致する行を検索
        target_rows = []
        for i in range(data_start_row, len(result_df)):
            if i >= len(result_df) or region_col_idx >= len(result_df.columns) or industry_col_idx >= len(result_df.columns):
                continue

            current_region = str(result_df.iloc[i, region_col_idx]) if pd.notna(result_df.iloc[i, region_col_idx]) else ""
            current_industry = str(result_df.iloc[i, industry_col_idx]) if pd.notna(result_df.iloc[i, industry_col_idx]) else ""

            if current_region == region and current_industry in industries_to_integrate:
                target_rows.append(i)

        if not target_rows:
            continue

        # 統合行の位置（最初のターゲット行を使用）
        insert_row = min(target_rows)

        # 産業名を更新
        result_df.iloc[insert_row, industry_col_idx] = new_industry_name

        # データ列の値を合計
        for col in range(4, len(result_df.columns)):
            sum_value = 0
            for row in target_rows:
                if row < len(result_df) and col < len(result_df.columns):
                    val = result_df.iloc[row, col]
                    if isinstance(val, (int, float)) and not pd.isna(val):
                        sum_value += val

            if insert_row < len(result_df) and col < len(result_df.columns):
                result_df.iloc[insert_row, col] = sum_value

        # 冗長な行を削除（最初の行を保持）
        rows_to_drop = target_rows[1:]
        if rows_to_drop:
            result_df = result_df.drop(index=rows_to_drop).reset_index(drop=True)

    return result_df

def integrate_industries_column(df, region, integration_operations, region_row_idx=3, industry_row_idx=5, data_start_col=4):
    """列方向で産業を統合（1対1の名称変更も含む）"""
    # 結果用データフレーム
    result_df = df.copy()

    # 各統合操作を処理
    for operation in integration_operations:
        industries_to_integrate = operation["industries_to_integrate"]
        new_industry_name = operation["new_industry_name"]

        # 地域と産業の条件に一致する列を検索
        target_cols = []
        for i in range(data_start_col, len(result_df.columns)):
            if (i < len(result_df.columns) and
                region_row_idx < len(result_df) and
                industry_row_idx < len(result_df)):

                region_val = str(result_df.iloc[region_row_idx, i]) if pd.notna(result_df.iloc[region_row_idx, i]) else ""
                industry_val = str(result_df.iloc[industry_row_idx, i]) if pd.notna(result_df.iloc[industry_row_idx, i]) else ""

                if region_val == region and industry_val in industries_to_integrate:
                    target_cols.append(i)

        if not target_cols:
            continue

        # 統合列の位置（最初のターゲット列を使用）
        insert_col = min(target_cols)

        # 産業名を更新
        if industry_row_idx < len(result_df) and insert_col < len(result_df.columns):
            result_df.iloc[industry_row_idx, insert_col] = new_industry_name

        # データ行の値を合計
        for row in range(6, len(result_df)):
            if row >= len(result_df):
                continue

            sum_value = 0
            for col in target_cols:
                if col < len(result_df.columns):
                    val = result_df.iloc[row, col]
                    if isinstance(val, (int, float)) and not pd.isna(val):
                        sum_value += val

            if insert_col < len(result_df.columns):
                result_df.iloc[row, insert_col] = sum_value

        # 冗長な列を削除（最初の列を保持）
        cols_to_drop = target_cols[1:]
        if cols_to_drop:
            cols_to_keep = [i for i in range(len(result_df.columns)) if i not in cols_to_drop]
            result_df = result_df.iloc[:, cols_to_keep]

    return result_df

def process_io_table(file_path, integration_operations, industries_to_remove):
    """与えられた統合操作と削除対象産業で産業連関表ファイルを処理"""
    try:
        # 最初のシートを読み込む（デフォルト動作）
        df = pd.read_excel(file_path, header=None)
        print(f"{file_path} の最初のシートを読み込みました")
    except Exception as e:
        print(f"{file_path} の読み込みエラー: {e}")
        return None

    # 地域を定義
    regions = ['北海道', '東北', '関東', '中部', '近畿', '中国', '四国', '九州', '沖縄', '地域計']

    # まず不要な産業を削除
    for region in regions:
        df = remove_industries_row(df, region, industries_to_remove)
        df = remove_industries_column(df, region, industries_to_remove)

    # その後に産業の統合を処理
    for region in regions:
        df = integrate_industries_row(df, region, integration_operations)
        df = integrate_industries_column(df, region, integration_operations)

    # 保存前にタブ文字を削除
    df = df.applymap(lambda x: x.replace('\t', '') if isinstance(x, str) else x)

    # B列が「地域計」かつD列が「地域内生産額」の行を探し、それより下の行を削除
    drop_row_index = -1
    # Assuming B column is index 1 and D column is index 3 (0-indexed)
    b_col_idx = 1
    d_col_idx = 3

    for i in range(len(df)):
        if i < len(df) and b_col_idx < len(df.columns) and d_col_idx < len(df.columns):
            b_val = str(df.iloc[i, b_col_idx]) if pd.notna(df.iloc[i, b_col_idx]) else ""
            d_val = str(df.iloc[i, d_col_idx]) if pd.notna(df.iloc[i, d_col_idx]) else ""

            if b_val == '地域計' and d_val == '地域内生産額':
                drop_row_index = i
                break

    # Drop rows below the identified index
    if drop_row_index != -1 and drop_row_index + 1 < len(df):
        df = df.iloc[:drop_row_index + 1].reset_index(drop=True)
        print(f"💡 B列が'地域計'かつD列が'地域内生産額'の行以下の行を削除しました。")


    return df

def main():
    """統合プロセスを実行するメイン関数"""

    # マッピングファイルの読み込み（MAPPING_DIRを参照）
    mapping_path = MAPPING_DIR / '産業分類統一対応表_S60_H2_H7_H17.xlsx'
    mapping_df = load_mapping_file(mapping_path)

    if mapping_df is None:
        print(f"マッピングファイルの読み込みに失敗しました: {mapping_path}")
        return

    # 各年の操作と削除対象を作成
    s60_operations, s60_to_remove = create_integration_operations(mapping_df, 'S60')
    h2_operations, h2_to_remove = create_integration_operations(mapping_df, 'H2')
    h7_operations, h7_to_remove = create_integration_operations(mapping_df, 'H7')
    h17_operations, h17_to_remove = create_integration_operations(mapping_df, 'H17')

    # 各ファイルを処理（RAW_DATA_DIRから読み込み、INTERIM_DIRへ保存）
    years = [
        {
            'file': RAW_DATA_DIR / 'S60.xlsx',
            'operations': s60_operations,
            'to_remove': s60_to_remove,
            'output': INTERIM_DIR / 'S60_integrated.xlsx'
        },
        {
            'file': RAW_DATA_DIR / 'H2.xlsx',
            'operations': h2_operations,
            'to_remove': h2_to_remove,
            'output': INTERIM_DIR / 'H2_integrated.xlsx'
        },
        {
            'file': RAW_DATA_DIR / 'H7.xlsx',
            'operations': h7_operations,
            'to_remove': h7_to_remove,
            'output': INTERIM_DIR / 'H7_integrated.xlsx'
        },
        {
            'file': RAW_DATA_DIR / 'H17.xlsx',
            'operations': h17_operations,
            'to_remove': h17_to_remove,
            'output': INTERIM_DIR / 'H17_integrated.xlsx'
        }
    ]

    for year in years:
        file_path = str(year['file'])
        output_path = str(year['output'])

        print(f"\n{os.path.basename(file_path)} を処理中...")

        if not os.path.exists(file_path):
            print(f"警告: ファイル {file_path} が見つかりません。スキップします。")
            continue

        result = process_io_table(file_path, year['operations'], year['to_remove'])

        if result is not None:
            result.to_excel(output_path, index=False, header=False)
            print(f"✓ 完了: {os.path.basename(output_path)}")
        else:
            print(f"✗ 失敗: {os.path.basename(file_path)}")

    print("\n全ての処理が完了しました。")

# メイン関数を実行
if __name__ == "__main__":
    main()


S60.xlsx を処理中...
S60.xlsx の最初のシートを読み込みました


/tmp/ipython-input-3878765919.py:243: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.replace('\t', '') if isinstance(x, str) else x)


S60.xlsx の処理が完了し、S60_integrated.xlsx として保存されました。

H2.xlsx を処理中...
H2.xlsx の最初のシートを読み込みました


/tmp/ipython-input-3878765919.py:243: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.replace('\t', '') if isinstance(x, str) else x)


H2.xlsx の処理が完了し、H2_integrated.xlsx として保存されました。

H7.xlsx を処理中...
H7.xlsx の最初のシートを読み込みました


/tmp/ipython-input-3878765919.py:243: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.replace('\t', '') if isinstance(x, str) else x)


💡 B列が'地域計'かつD列が'地域内生産額'の行以下の行を削除しました。
H7.xlsx の処理が完了し、H7_integrated.xlsx として保存されました。

H17.xlsx を処理中...
H17.xlsx の最初のシートを読み込みました


/tmp/ipython-input-3878765919.py:243: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.replace('\t', '') if isinstance(x, str) else x)


💡 B列が'地域計'かつD列が'地域内生産額'の行以下の行を削除しました。
H17.xlsx の処理が完了し、H17_integrated.xlsx として保存されました。

全ての処理が完了しました。


# 列差分検出

In [ ]:
# ==========================================
# 4. データの整合性検証（列差分検出）
# ==========================================
print(f"\n{'='*60}")
print("列名（部門構成）の整合性チェックを実行中...")
print(f"{'='*60}")

column_names_by_year = {}

# YEARS_CONFIG をループして、INTERIM_DIR 内の処理済みファイルを読み込む
for year, files in YEARS_CONFIG.items():
    # 統合後のファイル（例: S60_integrated.xlsx）のパスを作成
    file_path = INTERIM_DIR / files["integrated"]

    if not file_path.exists():
        print(f"警告: 検証対象のファイルが見つかりません: {file_path}")
        continue

    try:
        # 前処理コードでラベルは6行目(header=5)に設定されている前提
        df = pd.read_excel(file_path, header=5)
        column_names_by_year[year] = df.columns.tolist()
    except Exception as e:
        print(f"エラー: {year} の読み込み中に問題が発生しました: {e}")

# 比較ロジックはそのまま維持
years_list = list(column_names_by_year.keys())
differences = {}

for i in range(len(years_list)):
    for j in range(i + 1, len(years_list)):
        year1 = years_list[i]
        year2 = years_list[j]
        cols1 = set(column_names_by_year[year1])
        cols2 = set(column_names_by_year[year2])

        diff1 = list(cols1 - cols2)
        diff2 = list(cols2 - cols1)

        if diff1 or diff2:
            differences[f'{year1} vs {year2}'] = {
                f'Only in {year1}': diff1,
                f'Only in {year2}': diff2
            }

# 結果の出力
if differences:
    for comparison, diff_details in differences.items():
        print(f"\n[差分検出] {comparison}:")
        for category, cols in diff_details.items():
            print(f"  {category}: {cols}")
    print("\n部門構成に不一致があります。マッピング表を再確認してください。")
else:
    print("\n全てのファイルで列名（部門構成）が一致していることを確認しました。")

全てのファイルで列名に差分はありませんでした。
